# #8.3 File Search — RAG를 직접 돌려보기

**목적**: FileSearch(= RAG)를 단계별로 직접 실행해본다. `goals.txt`를 Vector Store에 넣고 → 검색 → 에이전트가 참조하는 것까지.

> ⚠️ 이 노트북은 **실제 OpenAI Vector Store를 만든다** (계정 리소스 + 소액 비용). 맨 아래 **삭제 셀**로 정리한다.

**시험과 연결** ([기말정리](../../../../../../지능로봇개론/기말정리.md)):
- Vector Store = #8 RAG '의미 기반 장기 기억'
- upload_and_poll = #3 랭체인 5요소(로드→청크→임베딩→저장)
- search = #8 유사도 검색(Retriever) / #9 임베딩

In [1]:
from pathlib import Path
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents import Agent, Runner, FileSearchTool, WebSearchTool

load_dotenv()
print("키 있음 ✅" if os.getenv("OPENAI_API_KEY") else "키 없음 ❌ — .env 확인")
client = OpenAI()

키 있음 ✅


## 1) Vector Store 만들기 = RAG 저장소

내 문서를 '의미로 검색 가능한' 형태로 담아둘 통. (아직 비어 있음 → 다음 셀에서 채움)

In [2]:
vs = client.vector_stores.create(name="life-coach-goals-nb")
print("Vector Store id:", vs.id)
print("파일 수:", vs.file_counts)

Vector Store id: vs_6a31f0d9a11c81919921ea03c86343e3
파일 수: FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0)


## 2) goals.txt 업로드 + 임베딩

`upload_and_poll` 한 줄이 **로드 → 청크 분할 → 임베딩 → 저장**(시험 #3 랭체인 5요소)을 다 하고, 끝날 때까지 기다린다(`poll`).

업로드 후 `files.list`로 상태가 `completed`인지 확인.

In [3]:
goals = Path("goals.txt")
client.vector_stores.files.upload_and_poll(
    vector_store_id=vs.id,
    file=(goals.name, goals.read_bytes()),
)

files = client.vector_stores.files.list(vector_store_id=vs.id)
for f in files.data:
    print(f.id, "→", f.status)  # completed 면 임베딩 끝

file-Rc9quvwULBtfHrCD3BKkzp → completed


## 3) Retriever만 떼서 보기 — LLM 없이 '유사도 검색'

에이전트 붙이기 전에, **검색 단계만** 직접 호출해본다. 질문을 벡터로 바꿔 의미가 가까운 청크를 돌려준다(시험 #8 유사도 검색).

👉 질문에 '운동'이라는 단어가 정확히 없어도 의미로 찾아오는지 봐라.

In [4]:
res = client.vector_stores.search(vector_store_id=vs.id, query="몸 만들려고 세운 계획이 뭐였지?")
for hit in res.data:
    score = round(hit.score, 3)
    text = hit.content[0].text if hit.content else ""
    print(f"[유사도 {score}] {text[:120]}...")
    print("-" * 60)

[유사도 0.308] # 나의 개인 목표 (2026)

## 운동
- 주 3회 이상 운동한다.
- 러닝은 회당 20분 이상.
- PT 비용이 부담돼서, PT 없이 헬스장에서 혼자 운동하고 싶다.
  올바른 호흡법과 기구(머신) 사용법을 ...
------------------------------------------------------------


## 4) FileSearchTool 붙인 에이전트에게 물어보기

이제 도구로 붙인다. 에이전트가 알아서 위 검색을 호출해 내 목표를 근거로 답한다.

- `vector_store_ids`는 **리스트** / `max_num_results=3` → 상위 3개 청크만

In [5]:
agent = Agent(
    name="Life Coach",
    instructions="너는 라이프 코치야. 사용자의 목표·기록을 물으면 FileSearchTool로 목표 문서를 찾아 근거로 답해. 한국어 존댓말.",
    tools=[FileSearchTool(vector_store_ids=[vs.id], max_num_results=3)],
)

result = await Runner.run(agent, "내 운동 목표가 뭐였고, 어떻게 하면 잘 지킬 수 있을까?")
print(result.final_output)

운동 목표는 이렇게 적혀 있습니다: 주 3회 이상 운동하고, 러닝은 회당 20분 이상 하며, PT 없이 헬스장에서 혼자 운동하면서 올바른 호흡법과 기구 사용법을 익히는 것이 목표예요 

잘 지키려면 이렇게 해보시면 좋습니다:
- 주간 고정 일정으로 3회 먼저 박아두기
- 러닝은 “20분만”으로 기준을 단순하게 두기
- 헬스장 운동은 매번 같은 루틴으로 시작하기
- 기구 사용법은 한 번에 하나씩 익히기
- PT 대신 메모나 영상으로 자세를 기록하며 점검하기

원하시면 제가 이 목표를 바탕으로 “주 3회 운동 루틴”까지 바로 짜드릴게요.


## 5) 이벤트로 `file_search_call` 흔적 보기

스트리밍하면서 **도구 호출 이벤트**를 찍어본다. `response.file_search_call.*` 가 보이면 = 에이전트가 목표 문서를 검색한 것. (app.py의 STATUS_MESSAGES·paint_history가 잡는 그 이벤트)

In [6]:
runner = Runner.run_streamed(agent, "내 수면 목표는 뭐야? 짧게.")
async for event in runner.stream_events():
    if event.type != "raw_response_event":
        continue
    t = event.data.type
    if "file_search_call" in t:
        print(f"\n📂 [{t}]")
    elif t == "response.output_text.delta":
        print(event.data.delta, end="")


📂 [response.file_search_call.in_progress]

📂 [response.file_search_call.searching]

📂 [response.file_search_call.completed]
매일 밤 12시 전에 취침하고 아침 7시에 기상하는 거예요. 1순위는 수면 정상화예요. 

## 6) FileSearch + WebSearch 결합 (과제 요구)

두 도구를 다 붙이면: **내 목표(파일)** + **검증된 방법(웹)**을 합쳐 개인화 추천. 이벤트에 📂와 🔍가 둘 다 나오는지 봐라.

In [7]:
agent2 = Agent(
    name="Life Coach",
    instructions="목표는 FileSearch로, 검증된 방법은 WebSearch로 찾아 합쳐서 조언해. 한국어 존댓말.",
    tools=[
        FileSearchTool(vector_store_ids=[vs.id], max_num_results=3),
        WebSearchTool(),
    ],
)

runner = Runner.run_streamed(agent2, "내 운동 목표에 맞는, 효과 검증된 러닝 초보 루틴을 추천해줘.")
async for event in runner.stream_events():
    if event.type != "raw_response_event":
        continue
    t = event.data.type
    if "file_search_call" in t:
        print(f"\n📂 [{t}]")
    elif "web_search_call" in t:
        print(f"\n🔍 [{t}]")
    elif t == "response.output_text.delta":
        print(event.data.delta, end="")


📂 [response.file_search_call.in_progress]

📂 [response.file_search_call.searching]

📂 [response.file_search_call.completed]

🔍 [response.web_search_call.in_progress]

🔍 [response.web_search_call.searching]

🔍 [response.web_search_call.completed]
네. 현재 목표를 보면 주 3회 이상, 러닝은 회당 20분 이상, 그리고 PT 없이 혼자 꾸준히 할 수 있는 루틴이 가장 잘 맞습니다 리기 인터벌”이 가장 무난합니다. Mayo Clinic도 초보자는 천천히 시작해서 점차 시간/속도를 늘리고, 인터벌 방식이 심폐지구력 향상에 도움이 된다고 안내합니다 ([mayoclinic.org](https://www.mayoclinic.org/healthy-lifestyle/fitness/in-depth/walking/art-20046261?utm_source=openai))

추천 루틴(4주):
- 주 3회
- 각 20~30분
- 강도: “숨은 차지만 대화는 짧게 가능한 정도”

1주차
- 5분 걷기
- 1분 달리기 + 2분 걷기 × 5세트
- 5분 걷기

2주차
- 5분 걷기
- 90초 달리기 + 2분 걷기 × 5세트
- 5분 걷기

3주차
- 5분 걷기
- 2분 달리기 + 90초 걷기 × 5세트
- 5분 걷기

4주차
- 5분 걷기
- 3분 달리기 + 90초 걷기 × 4세트
- 5분 걷기

운동 전후 팁:
- 시작 전 5분 워밍업, 끝나면 5분 쿨다운
- 통증이 아니라 “힘듦” 수준에서 멈추기
- 주간 총량을 조금씩만 늘리기 ([mayoclinic.org](https://www.mayoclinic.org/healthy-lifestyle/fitness/in-depth/walking/art-20046261?utm_source=openai))

당장 가장 중요한 기준은 “완주 가능”입니다. 

## 7) 정리 — Vector Store 삭제

실험용으로 만든 통을 지운다(계정 청소). 더 만져보고 싶으면 나중에 실행.

In [8]:
#주석 풀고 실행하면 삭제
client.vector_stores.delete(vs.id)
print("삭제됨:", vs.id)
print("삭제하려면 윗줄 주석 해제")

삭제됨: vs_6a31f0d9a11c81919921ea03c86343e3
삭제하려면 윗줄 주석 해제
